# Merge Refined Themes with Camilla's Transcript Preprocessing

Camilla has since updated her `preprocessing_pipeline.ipynb` to preprocess the **transcript** column (full talk text) instead. Her output `preprocessed_full.csv` has two new text columns:
- `clean_tokens` — list of preprocessed tokens from the transcript
- `clean_text_joined` — those tokens joined into a space-separated string

This notebook merges those columns into the themed dataset to produce an updated `ted_master.csv` for sentiment analysis.
**Inputs**
- `ted_themes_v2.csv`
- `preprocessed_full.csv`
-
**Output**
- `ted_master.csv`

## 1. Setup

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 80)

def find_first(filename, candidates):
    for p in candidates:
        if Path(p).exists():
            return Path(p)
    raise FileNotFoundError(f'Could not find {filename!r}. Tried: {candidates}')

THEMES_PATH = find_first(
    'ted_themes_v2',
    [
        '../data/processed/ted_themes_v2.csv',
        '../data/processed/ted_themes_v2.csv.zip',
        '../data/ted_themes_v2.csv',
        '../data/ted_themes_v2.csv.zip',
        'ted_themes_v2.csv',
        'ted_themes_v2.csv.zip',
    ],
)
PREPROCESSED_PATH = find_first(
    'preprocessed_full.csv',
    [
        '../results/preprocessed_full.csv',
        '../data/processed/preprocessed_full.csv',
        'preprocessed_full.csv',
    ],
)
OUTPUT_DIR = THEMES_PATH.parent

print(f'Themes file:        {THEMES_PATH}')
print(f'Preprocessing file: {PREPROCESSED_PATH}')
print(f'Output dir:         {OUTPUT_DIR}')

Themes file:        ../data/processed/ted_themes_v2.csv
Preprocessing file: ../results/preprocessed_full.csv
Output dir:         ../data/processed


## 2. Load both inputs

In [2]:
themes = pd.read_csv(THEMES_PATH)
print(f'Themed dataset: {len(themes):,} rows × {themes.shape[1]} columns')
print('  columns:', list(themes.columns))

Themed dataset: 2,453 rows × 23 columns
  columns: ['comments', 'description', 'duration', 'event', 'film_date', 'languages', 'main_speaker', 'name', 'num_speaker', 'published_date', 'ratings', 'related_talks', 'speaker_occupation', 'tags', 'title', 'url', 'views', 'transcript', 'transcript_len_words', 'transcript_len_chars', 'num_tags', 'num_themes_multi', 'theme']


In [3]:
preproc = pd.read_csv(PREPROCESSED_PATH)
print(f'Preprocessed: {len(preproc):,} rows × {preproc.shape[1]} columns')
print('  columns:', list(preproc.columns))

Preprocessed: 2,467 rows × 7 columns
  columns: ['name', 'transcript', 'clean_tokens', 'clean_text_joined', 'tags', 'views', 'comments']


## 3. Decide on the join key

Camilla's preprocessed file does not contain `url`, but it does contain `name`. We join on `name`, after sanity-checking uniqueness and overlap.

In [4]:
print(f"Themed 'name' unique:    {themes['name'].is_unique}")
print(f"  duplicates: {themes['name'].duplicated().sum()}")
print(f"\nPreproc 'name' unique:   {preproc['name'].is_unique}")
print(f"  duplicates: {preproc['name'].duplicated().sum()}")

overlap = set(themes['name']) & set(preproc['name'])
print(f'\nNames in both files:    {len(overlap):,}')
print(f'  In themes only:        {len(set(themes["name"]) - set(preproc["name"])):,}')
print(f'  In preprocessed only:  {len(set(preproc["name"]) - set(themes["name"])):,}')

Themed 'name' unique:    True
  duplicates: 0

Preproc 'name' unique:   False
  duplicates: 3

Names in both files:    2,453
  In themes only:        0
  In preprocessed only:  11


## 4. Merge

We keep only the new text columns from Camilla's file plus the join key. Her copies of `tags`, `views`, and `comments` are dropped to avoid column-name clashes — the themed dataset is the authoritative source for those fields. Any older description-based `clean_text` column dropped.

In [5]:
preproc_minimal = preproc[['name', 'clean_tokens', 'clean_text_joined']].copy()
themes_no_old = themes.drop(columns=['clean_text'], errors='ignore')
master = themes_no_old.merge(preproc_minimal, on='name', how='inner')

print(f'Merged master: {len(master):,} rows × {master.shape[1]} columns')
print(f'  Lost: {len(themes) - len(master):,} rows (no transcript-based clean_text available)')
print(f'\nFinal columns:\n{list(master.columns)}')

Merged master: 2,456 rows × 25 columns
  Lost: -3 rows (no transcript-based clean_text available)

Final columns:
['comments', 'description', 'duration', 'event', 'film_date', 'languages', 'main_speaker', 'name', 'num_speaker', 'published_date', 'ratings', 'related_talks', 'speaker_occupation', 'tags', 'title', 'url', 'views', 'transcript', 'transcript_len_words', 'transcript_len_chars', 'num_tags', 'num_themes_multi', 'theme', 'clean_tokens', 'clean_text_joined']


In [6]:
master[['name', 'theme', 'clean_text_joined']].head(3)

,name,theme,clean_text_joined
0,Ken Robinson: Do schools kill creativity?,education,good morning you?(laughter)it great blow away thing fact leaving.(laughter)t...
1,Al Gore: Averting the climate crisis,environment,thank chris truly great honor opportunity come stage twice extremely gratefu...
2,David Pogue: Simplicity sells,unclassified,music sound silence simon garfunkel)hello voice mail old friend.(laughter)i'...


In [7]:
print('Theme distribution after merge:')
print(master['theme'].value_counts())

Theme distribution after merge:
theme
unclassified           481
environment            401
business               386
psychology             353
politics_governance    331
culture_society        253
education              251
Name: count, dtype: int64


## 5. Save

In [8]:
out_path = OUTPUT_DIR / 'ted_master.csv'
master.to_csv(out_path, index=False)
print(f'Saved -> {out_path}')
print(f'   {len(master):,} rows × {master.shape[1]} columns')
print('\nFor sentiment analysis, use `clean_text_joined`')
print('(or `clean_tokens` if your sentiment library expects a token list).')

Saved -> ../data/processed/ted_master.csv
   2,456 rows × 25 columns

For sentiment analysis, use `clean_text_joined`
(or `clean_tokens` if your sentiment library expects a token list).
